In [3]:
pip install --upgrade pip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline # Standar MLOps untuk SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

print("TRAINING EWS TAHAP 1: MLOps PIPELINE & BALANCED TUNING")

df_master = pd.read_csv('../data/data_master_clustered.csv')

# Gabung kelas jadi biner (0 = Aman, 1 = Bahaya)
def gabung_jadi_biner(status):
    return 0 if status == 0 else 1

df_master['target_biner'] = df_master['target_ews'].apply(gabung_jadi_biner)

fitur_ews = [
    'Rainfall', 'SPI - 3 months', 'Temperature', 'Water Satisfaction Index (WSI)', 
    'Solar Radiation', 'Soil Moisture (gapfilled historical time series)', 
    'FPAR', 'FPAR - zscore', 'month_extracted'
]
target_ews = 'target_biner'

# RUANG PENCARIAN (Biar mesin yang cari batas ideal, bukan kita tebak manual)
# awalan 'xgb__' karena parameter ini milik XGBoost di dalam Pipeline
param_distributions = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.05, 0.1, 0.2],
    'xgb__scale_pos_weight': [1.0, 1.2, 1.5], # Mesin akan mencoba seberapa "parno" yang paling pas
    'xgb__subsample': [0.8, 1.0]
}

for cluster_id in sorted(df_master['Cluster_Wilayah'].unique()):
    print(f"\n==================================================")
    print(f"MEMPROSES CLUSTER {cluster_id}")
    print(f"==================================================")
    
    df_cluster = df_master[df_master['Cluster_Wilayah'] == cluster_id].copy()
    
    X = df_cluster[fitur_ews]
    y = df_cluster[target_ews]
    
    # 1. Split Data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # 2. MEMBANGUN MLOps PIPELINE
    # Alur urut: Data Latih -> di-SMOTE -> di-Scale -> masuk XGBoost
    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('xgb', xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42, n_jobs=-1))
    ])
    
    # 3. HYPERPARAMETER TUNING DENGAN PIPELINE
    print("Mencari parameter paling seimbang (F1-Score)... ")
    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=10, 
        scoring='f1', # Targetkan F1-Score agar Precision dan Recall sama-sama tinggi
        cv=3, 
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Proses Training (SMOTE hanya akan terjadi pada data latih di dalam lipatan CV)
    random_search.fit(X_train, y_train)
    
    # 4. EVALUASI MODEL TERBAIK
    best_pipeline = random_search.best_estimator_
    print(f"Parameter Juara Cluster {cluster_id}:")
    # Hanya menampilkan parameter milik XGBoost agar rapi
    for param_name in sorted(param_distributions.keys()):
        print(f"  - {param_name.replace('xgb__', '')}: {random_search.best_params_[param_name]}")
    
    # Prediksi menggunakan test set (Pipeline akan otomatis melakukan scaling ke test set)
    y_pred = best_pipeline.predict(X_test)
    
    print(f"\n📊 Evaluasi Model Akhir Cluster {cluster_id}:")
    print(classification_report(y_test, y_pred, target_names=['Aman (0)', 'Bahaya (1)']))
    
    # 5. SIMPAN PIPELINE UTUH (Mempermudah saat dihubungkan ke Web nanti)
    joblib.dump(best_pipeline, f'../models_output/pipeline_ews_biner_cluster_{cluster_id}.pkl')
    print(f"Pipeline Utuh Cluster {cluster_id} Berhasil Disimpan!")

TRAINING EWS TAHAP 1: MLOps PIPELINE & BALANCED TUNING

MEMPROSES CLUSTER 0
Mencari parameter paling seimbang (F1-Score)... 
Parameter Juara Cluster 0:
  - learning_rate: 0.2
  - max_depth: 7
  - n_estimators: 300
  - scale_pos_weight: 1.2
  - subsample: 0.8

📊 Evaluasi Model Akhir Cluster 0:
              precision    recall  f1-score   support

    Aman (0)       0.93      0.89      0.91      2566
  Bahaya (1)       0.60      0.72      0.66       589

    accuracy                           0.86      3155
   macro avg       0.77      0.81      0.78      3155
weighted avg       0.87      0.86      0.86      3155

Pipeline Utuh Cluster 0 Berhasil Disimpan!

MEMPROSES CLUSTER 1
Mencari parameter paling seimbang (F1-Score)... 
Parameter Juara Cluster 1:
  - learning_rate: 0.2
  - max_depth: 7
  - n_estimators: 100
  - scale_pos_weight: 1.0
  - subsample: 0.8

📊 Evaluasi Model Akhir Cluster 1:
              precision    recall  f1-score   support

    Aman (0)       0.93      0.90      0.9

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

print("LAST PROSES: OPTIMAL THRESHOLD SEARCH DARI MODEL JUARA")

# Load data lagi untuk testing
df_master = pd.read_csv('../data/data_master_clustered.csv')

def gabung_jadi_biner(status):
    return 0 if status == 0 else 1

df_master['target_biner'] = df_master['target_ews'].apply(gabung_jadi_biner)

fitur_ews = [
    'Rainfall', 'SPI - 3 months', 'Temperature', 'Water Satisfaction Index (WSI)', 
    'Solar Radiation', 'Soil Moisture (gapfilled historical time series)', 
    'FPAR', 'FPAR - zscore', 'month_extracted'
]
target_ews = 'target_biner'

# Target Recall yang kita inginkan untuk EWS (misal: 0.85)
TARGET_RECALL = 0.85

for cluster_id in sorted(df_master['Cluster_Wilayah'].unique()):
    print(f"\n==================================================")
    print(f" MENCARI THRESHOLD EMAS CLUSTER {cluster_id}")
    print(f"==================================================")
    
    df_cluster = df_master[df_master['Cluster_Wilayah'] == cluster_id].copy()
    X = df_cluster[fitur_ews]
    y = df_cluster[target_ews]
    
    # Pastikan random_state sama persis (42) agar test set-nya sama dengan saat training
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # 1. LOAD MODEL PIPELINE TERBAIK (Tidak perlu training lagi!)
    try:
        best_pipeline = joblib.load(f'../models_output/pipeline_ews_biner_cluster_{cluster_id}.pkl')
    except FileNotFoundError:
        print(f"File model Cluster {cluster_id} tidak ditemukan. Pastikan sudah menjalankan kode sebelumnya.")
        continue
    
    # 2. Minta Probabilitas dari Model Juara
    y_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]
    
    # 3. Hitung Kurva Precision-Recall
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
    
    # 4. Cari Threshold di mana Recall mendekati target kita (>= 0.85)
    # Kita cari index di mana recall masih lebih besar atau sama dengan TARGET_RECALL
    optimal_idx = np.where(recalls >= TARGET_RECALL)[0][-1] 
    optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else thresholds[-1]
    
    print(f"Ditemukan! Untuk mencapai Recall {TARGET_RECALL}, Threshold yang digunakan adalah: {optimal_threshold:.3f}")
    
    # 5. Terapkan Threshold Emas tersebut
    y_pred_optimal = (y_pred_prob >= optimal_threshold).astype(int)
    
    print(f"\n Hasil Klasifikasi dengan Threshold {optimal_threshold:.3f}:")
    print(classification_report(y_test, y_pred_optimal, target_names=['Aman (0)', 'Bahaya (1)']))

LAST PROSES: OPTIMAL THRESHOLD SEARCH DARI MODEL JUARA

 MENCARI THRESHOLD EMAS CLUSTER 0
Ditemukan! Untuk mencapai Recall 0.85, Threshold yang digunakan adalah: 0.209

 Hasil Klasifikasi dengan Threshold 0.209:
              precision    recall  f1-score   support

    Aman (0)       0.96      0.77      0.86      2566
  Bahaya (1)       0.46      0.85      0.60       589

    accuracy                           0.79      3155
   macro avg       0.71      0.81      0.73      3155
weighted avg       0.87      0.79      0.81      3155


 MENCARI THRESHOLD EMAS CLUSTER 1
Ditemukan! Untuk mencapai Recall 0.85, Threshold yang digunakan adalah: 0.276

 Hasil Klasifikasi dengan Threshold 0.276:
              precision    recall  f1-score   support

    Aman (0)       0.95      0.79      0.86      1133
  Bahaya (1)       0.53      0.85      0.65       317

    accuracy                           0.80      1450
   macro avg       0.74      0.82      0.76      1450
weighted avg       0.86      0.8